# NASA FIRMS Data Ingestion & Curation
## Response Phase – Wildfire Management (Algeria)

### Overview

This notebook performs the **data ingestion** and **data curation** of wildfire hotspot detections collected from the **NASA FIRMS (Fire Information for Resource Management System)** API.

The data include both **historical wildfire events** and **Near Real-Time (NRT)** detections acquired from the **VIIRS S-NPP** and **MODIS** satellite sensors.

The objective is to produce a **clean, reliable, and well-documented Algerian wildfire hotspot dataset** that can be used in the **Response** phase of the AI Disaster Management pipeline.


---

## Objectives

This notebook performs the following tasks:

### Data Ingestion
- Retrieve historical wildfire events in Algeria.
- Retrieve Near Real-Time (NRT) wildfire detections.
- Collect data from both:
  - **VIIRS S-NPP**
  - **MODIS**
- Save the raw datasets for reproducibility.

### Data Curation
- Load and merge all raw datasets.
- Inspect and profile the collected data.
- Parse and standardize data types.
- Filter detections based on confidence levels.
- Remove non-vegetation fire detections.
- Handle sensor-specific differences (VIIRS vs MODIS).
- Remove duplicate detections(none found).
- Export the final curated dataset.

---

## Data Source

**NASA FIRMS API**

The downloaded datasets contain wildfire hotspot detections together with several attributes, including:

- Geographic coordinates (latitude, longitude)
- Acquisition date and time
- Satellite and sensor
- Fire Radiative Power (FRP)
- Detection confidence
- Day/Night observation
- Detection type (historical datasets)

The collected data are **tabular datasets (CSV)** and **not satellite images**.

---

## Output

The notebook produces:

- Raw wildfire datasets stored in `data/response/raw/`
- Curated wildfire dataset stored in `data/response/processed/`

---

## Notes

- Historical and Near Real-Time (NRT) datasets are processed separately during ingestion and combined during curation.
- VIIRS and MODIS use different confidence scales and some sensor-specific attributes, which are handled during the cleaning process.
- The resulting dataset will serve as the foundation for subsequent response-related analyses and integration with additional geospatial datasets in later stages of the project.
---

In [89]:
import os
import time
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from dotenv import load_dotenv

In [90]:
load_dotenv()

MAP_KEY = os.getenv("FIRMS_MAP_KEY")

if MAP_KEY is None:
    raise ValueError("FIRMS_MAP_KEY not found in .env")
 
RAW_DIR = Path("../data/Response/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
 
# Algeria 
ALGERIA_BBOX = "-8.67,18.96,11.99,37.09"
 
# Sensors available — use both for cross-validation
# VIIRS_SNPP_NRT: finer resolution (375m), available 2012–present
# MODIS_NRT: coarser (1km), available 2000–present (longer history)
NRT_SENSORS = {
    "VIIRS_SNPP_NRT": "viirs_snpp",
    "MODIS_NRT": "modis",
}

HISTORICAL_SENSORS = {
    "VIIRS_SNPP_SP": "viirs_snpp",
    "MODIS_SP": "modis",
}
 
BASE_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"

In [91]:
"""
    Build a FIRMS country API URL.
    
    Args:
        map_key:    personal FIRMS MAP_KEY
        sensor:     e.g. 'VIIRS_SNPP_NRT' or 'MODIS_NRT'
        bbox:       coordinates of the wanted area
        day_range:  number of days to pull (max 10 for NRT, up to 374 for archive)
        date:       start date string 'YYYY-MM-DD' (None = most recent day_range days)    
"""

def build_area_url(map_key, sensor, bbox, day_range, date=None):
    url = f"{BASE_URL}/{map_key}/{sensor}/{bbox}/{day_range}"
    if date:
        url += f"/{date}"
    return url

In [92]:
"""
    Pull FIRMS data for a geographic bounding box and return it as a DataFrame.

    Args:
        map_key:   Your FIRMS MAP_KEY.
        sensor:    e.g. 'VIIRS_SNPP_NRT' or 'MODIS_NRT'.
        bbox:      Bounding box as "minLon,minLat,maxLon,maxLat".
        day_range: Number of days to retrieve.
        date:      Optional start date ('YYYY-MM-DD') for historical queries.

    Returns:
        pandas.DataFrame
"""

def pull_firms_data(map_key, sensor, bbox, day_range, date=None):

    url = build_area_url(
        map_key,
        sensor,
        bbox,
        day_range,
        date
    )

    return pd.read_csv(url)

In [93]:
"""
    Pull FIRMS data for a known historical fire event within a geographic area.

    The FIRMS area endpoint accepts a bounding box, a start date,
    and a day range to retrieve historical fire detections.

    Args:
        map_key:    Your FIRMS MAP_KEY.
        sensor:     e.g. 'VIIRS_SNPP_NRT' or 'MODIS_NRT'.
        bbox:       Bounding box as "minLon,minLat,maxLon,maxLat".
        event_name: Label for the output file
                    (e.g. 'tizi_ouzou_bejaia_aug2021').
        start_date: Start date ('YYYY-MM-DD').
        n_days:     Number of days to retrieve.
"""

def pull_historical_event(map_key, sensor, bbox, event_name, start_date, n_days=14):
    
    df = pull_firms_data(map_key, sensor, bbox, n_days, start_date)
    if not df.empty:
        fname = RAW_DIR / f"firms_{sensor.lower()}_{event_name}.csv"
        df.to_csv(fname, index=False)
        print(f"  Saved → {fname}")
    return df

In [94]:
ALGERIA_FIRE_EVENTS = [
    {
        "name": "kabylie_aug_2021",
        "start_date": "2021-08-09",
        "n_days": 5,
        "notes": (
            "Major Kabylie fires (Tizi Ouzou, Béjaïa, Jijel, Bouira). "
            "Peak activity began on 9 August 2021."
        ),
    },
    {
        "name": "el_tarf_aug_2022",
        "start_date": "2022-08-17",
        "n_days": 5,
        "notes": (
            "Major northeastern Algeria fires affecting El Tarf, "
            "Souk Ahras and Guelma."
        ),
    },
    {
        "name": "bejaia_july_2023",
        "start_date": "2023-07-23",
        "n_days": 5,
        "notes": (
            "Major July 2023 fires affecting Béjaïa, Bouira, "
            "Tizi Ouzou and Jijel."
        ),
    },
]

In [95]:
print("=== Near Real-Time Data ===")

today = datetime.today().strftime("%Y%m%d")

for sensor in NRT_SENSORS:

    df = pull_firms_data(
        MAP_KEY,
        sensor,
        ALGERIA_BBOX,
        day_range=5
    )

    if not df.empty:
        filename = RAW_DIR / f"firms_{sensor.lower()}_nrt_{today}.csv"

        df.to_csv(filename, index=False)

        print(f"Saved: {filename}")

    time.sleep(1)

=== Near Real-Time Data ===
Saved: ../data/Response/raw/firms_viirs_snpp_nrt_nrt_20260704.csv
Saved: ../data/Response/raw/firms_modis_nrt_nrt_20260704.csv


In [96]:
print("=== Historical Fire Events ===")

for event in ALGERIA_FIRE_EVENTS:

    print(f"\nEvent: {event['name']}")

    for sensor in HISTORICAL_SENSORS:

        pull_historical_event(
            MAP_KEY,
            sensor,
            ALGERIA_BBOX,
            event_name=event["name"],
            start_date=event["start_date"],
            n_days=event["n_days"],
        )

        time.sleep(1)

=== Historical Fire Events ===

Event: kabylie_aug_2021
  Saved → ../data/Response/raw/firms_viirs_snpp_sp_kabylie_aug_2021.csv
  Saved → ../data/Response/raw/firms_modis_sp_kabylie_aug_2021.csv

Event: el_tarf_aug_2022
  Saved → ../data/Response/raw/firms_viirs_snpp_sp_el_tarf_aug_2022.csv
  Saved → ../data/Response/raw/firms_modis_sp_el_tarf_aug_2022.csv

Event: bejaia_july_2023
  Saved → ../data/Response/raw/firms_viirs_snpp_sp_bejaia_july_2023.csv
  Saved → ../data/Response/raw/firms_modis_sp_bejaia_july_2023.csv


# Loading all the raw extracted csv files into a single dataset df_raw

In [97]:
# Load all raw FIRMS CSVs from the raw directory
# Tag each row with its source file and pull type (nrt vs historical)

raw_files = list(RAW_DIR.glob("*.csv"))
print(f"Found {len(raw_files)} raw files:\n")

dfs = []
for f in raw_files:
    df = pd.read_csv(f)
    df["source_file"] = f.name
    df["pull_type"] = "nrt" if "nrt" in f.name.lower() else "historical"
    print(f"  {f.name} → {len(df)} rows")
    dfs.append(df)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows combined: {len(df_raw)}")

Found 8 raw files:

  firms_viirs_snpp_nrt_nrt_20260704.csv → 1655 rows
  firms_modis_sp_bejaia_july_2023.csv → 734 rows
  firms_modis_sp_el_tarf_aug_2022.csv → 539 rows
  firms_viirs_snpp_sp_bejaia_july_2023.csv → 2067 rows
  firms_modis_sp_kabylie_aug_2021.csv → 1842 rows
  firms_viirs_snpp_sp_kabylie_aug_2021.csv → 8499 rows
  firms_viirs_snpp_sp_el_tarf_aug_2022.csv → 2707 rows
  firms_modis_nrt_nrt_20260704.csv → 109 rows

Total rows combined: 18152


In [98]:
for file, df in zip(raw_files, dfs):
    print(f"\n{file.name}")
    print("-" * len(file.name))
    print(df.columns.tolist())


firms_viirs_snpp_nrt_nrt_20260704.csv
-------------------------------------
['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight', 'source_file', 'pull_type']

firms_modis_sp_bejaia_july_2023.csv
-----------------------------------
['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type', 'source_file', 'pull_type']

firms_modis_sp_el_tarf_aug_2022.csv
-----------------------------------
['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type', 'source_file', 'pull_type']

firms_viirs_snpp_sp_bejaia_july_2023.csv
----------------------------------------
['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'i

In [99]:
df_raw

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,source_file,pull_type,brightness,bright_t31,type
0,34.90565,8.53826,299.99,0.39,0.36,2026-06-30,120,N,VIIRS,n,2.0NRT,287.33,0.43,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
1,35.23201,8.17444,302.47,0.40,0.37,2026-06-30,120,N,VIIRS,n,2.0NRT,288.61,1.02,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
2,35.23264,8.17010,310.85,0.40,0.37,2026-06-30,120,N,VIIRS,n,2.0NRT,288.63,1.24,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
3,35.70739,4.47272,299.42,0.54,0.42,2026-06-30,120,N,VIIRS,n,2.0NRT,286.74,0.88,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
4,35.80626,-0.25923,321.66,0.32,0.55,2026-06-30,120,N,VIIRS,n,2.0NRT,293.09,2.42,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18147,35.46593,-1.12949,NaN,1.04,1.02,2026-07-04,1516,Aqua,MODIS,77,6.1NRT,NaN,16.33,D,firms_modis_nrt_nrt_20260704.csv,nrt,334.48,317.70,NaN
18148,35.47477,-1.13133,NaN,1.04,1.02,2026-07-04,1516,Aqua,MODIS,82,6.1NRT,NaN,23.07,D,firms_modis_nrt_nrt_20260704.csv,nrt,338.54,317.57,NaN
18149,35.47626,-1.12036,NaN,1.04,1.02,2026-07-04,1516,Aqua,MODIS,80,6.1NRT,NaN,19.28,D,firms_modis_nrt_nrt_20260704.csv,nrt,336.31,316.48,NaN
18150,35.90964,0.09219,NaN,1.14,1.06,2026-07-04,1516,Aqua,MODIS,56,6.1NRT,NaN,9.42,D,firms_modis_nrt_nrt_20260704.csv,nrt,326.86,312.90,NaN


In [100]:
print(f"Shape: {df_raw.shape}")
print(f"\nColumns:\n{list(df_raw.columns)}")
print(f"\nDtypes:\n{df_raw.dtypes}")
print(f"\nMissing values per column:")
print(df_raw.isnull().sum())
print(f"\nDuplicate rows: {df_raw.duplicated().sum()}")
print(f"\nInstrument breakdown:\n{df_raw['instrument'].value_counts()}")
print(f"\nPull type breakdown:\n{df_raw['pull_type'].value_counts()}")

Shape: (18152, 19)

Columns:
['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight', 'source_file', 'pull_type', 'brightness', 'bright_t31', 'type']

Dtypes:
latitude       float64
longitude      float64
bright_ti4     float64
scan           float64
track          float64
acq_date           str
acq_time         int64
satellite          str
instrument         str
confidence      object
version         object
bright_ti5     float64
frp            float64
daynight           str
source_file        str
pull_type          str
brightness     float64
bright_t31     float64
type           float64
dtype: object

Missing values per column:
latitude           0
longitude          0
bright_ti4      3224
scan               0
track              0
acq_date           0
acq_time           0
satellite          0
instrument         0
confidence         0
version            0
bright_ti5      3224
f

# Changing the date column's type into datetime 

In [101]:
df_raw["acq_date"] = pd.to_datetime(df_raw["acq_date"])

print(f"Date range: {df_raw['acq_date'].min()} → {df_raw['acq_date'].max()}")
print(f"\nDetections per source event:")
print(df_raw.groupby("source_file").size().to_string())

Date range: 2021-08-09 00:00:00 → 2026-07-04 00:00:00

Detections per source event:
source_file
firms_modis_nrt_nrt_20260704.csv             109
firms_modis_sp_bejaia_july_2023.csv          734
firms_modis_sp_el_tarf_aug_2022.csv          539
firms_modis_sp_kabylie_aug_2021.csv         1842
firms_viirs_snpp_nrt_nrt_20260704.csv       1655
firms_viirs_snpp_sp_bejaia_july_2023.csv    2067
firms_viirs_snpp_sp_el_tarf_aug_2022.csv    2707
firms_viirs_snpp_sp_kabylie_aug_2021.csv    8499


# Confidence filter 
#### VIIRS and MODIS use different confidence formats
#### Low-confidence detections are likely false positives 


In [102]:
viirs = df_raw[df_raw["instrument"] == "VIIRS"].copy()
modis = df_raw[df_raw["instrument"] == "MODIS"].copy()

print(f"Before filtering — VIIRS: {len(viirs)} rows, MODIS: {len(modis)} rows")
print(f"\nVIIRS confidence values:\n{viirs['confidence'].value_counts()}")
print(f"\nMODIS confidence range: {modis['confidence'].min()} → {modis['confidence'].max()}")


Before filtering — VIIRS: 14928 rows, MODIS: 3224 rows

VIIRS confidence values:
confidence
n    10837
h     2161
l     1930
Name: count, dtype: int64

MODIS confidence range: 0 → 100


In [103]:
# VIIRS: keep nominal and high only
viirs_clean = viirs[viirs["confidence"].astype(str).isin(["n", "h"])]
print(f"\nVIIRS after filter: {len(viirs_clean)} rows (dropped {len(viirs) - len(viirs_clean)})")

# MODIS: keep confidence >= 50
modis["confidence"] = pd.to_numeric(modis["confidence"], errors="coerce")
modis_clean = modis[modis["confidence"] >= 50]
print(f"MODIS after filter: {len(modis_clean)} rows (dropped {len(modis) - len(modis_clean)})")



VIIRS after filter: 12998 rows (dropped 1930)
MODIS after filter: 2991 rows (dropped 233)


In [104]:
df_filtered = pd.concat([viirs_clean, modis_clean], ignore_index=True)
print(f"\nCombined after confidence filter: {len(df_filtered)} rows")


Combined after confidence filter: 15989 rows


# The 'type' column identifies what kind of source the detection is:
   ##### 0 = presumed vegetation fire  
   ##### 1 = active volcano
   ##### 2 = other static land source (e.g. industrial flares, gas burns)
   ##### 3 = offshore detection
#### Keep vegetation fires.
#### NRT detections have no 'type' field, so keep missing values as well.


In [105]:
df_raw["type"].unique()

array([nan,  2.,  0.,  3.])

In [106]:
print(f"Type distribution:\n{df_filtered['type'].value_counts()}")

before = len(df_filtered)

df_filtered = df_filtered[
    (df_filtered["type"] == 0) |
    (df_filtered["type"].isna())
]

print(f"After vegetation-fire filter: {len(df_filtered)} rows")

Type distribution:
type
0.0    11169
2.0     3177
3.0       11
Name: count, dtype: int64
After vegetation-fire filter: 12801 rows


# Satellite orbits overlap — the same fire pixel can appear in consecutive
#### overpasses or in overlapping tiles from the same pass
#### Deduplicate on: location + date + time + instrument

In [107]:
duplicate_mask = df_filtered.duplicated(
    subset=["latitude", "longitude", "acq_date", "acq_time", "instrument"],
    keep=False
)

print(f"Number of duplicated rows: {duplicate_mask.sum()}")

Number of duplicated rows: 0


In [108]:
df_filtered

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,source_file,pull_type,brightness,bright_t31,type
0,34.90565,8.53826,299.99,0.39,0.36,2026-06-30,120,N,VIIRS,n,2.0NRT,287.33,0.43,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
1,35.23201,8.17444,302.47,0.40,0.37,2026-06-30,120,N,VIIRS,n,2.0NRT,288.61,1.02,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
2,35.23264,8.17010,310.85,0.40,0.37,2026-06-30,120,N,VIIRS,n,2.0NRT,288.63,1.24,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
3,35.70739,4.47272,299.42,0.54,0.42,2026-06-30,120,N,VIIRS,n,2.0NRT,286.74,0.88,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
4,35.80626,-0.25923,321.66,0.32,0.55,2026-06-30,120,N,VIIRS,n,2.0NRT,293.09,2.42,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15984,35.46593,-1.12949,NaN,1.04,1.02,2026-07-04,1516,Aqua,MODIS,77,6.1NRT,NaN,16.33,D,firms_modis_nrt_nrt_20260704.csv,nrt,334.48,317.70,NaN
15985,35.47477,-1.13133,NaN,1.04,1.02,2026-07-04,1516,Aqua,MODIS,82,6.1NRT,NaN,23.07,D,firms_modis_nrt_nrt_20260704.csv,nrt,338.54,317.57,NaN
15986,35.47626,-1.12036,NaN,1.04,1.02,2026-07-04,1516,Aqua,MODIS,80,6.1NRT,NaN,19.28,D,firms_modis_nrt_nrt_20260704.csv,nrt,336.31,316.48,NaN
15987,35.90964,0.09219,NaN,1.14,1.06,2026-07-04,1516,Aqua,MODIS,56,6.1NRT,NaN,9.42,D,firms_modis_nrt_nrt_20260704.csv,nrt,326.86,312.90,NaN


# Sort by date and reset index

In [109]:
df_clean = df_filtered.sort_values("acq_date").reset_index(drop=True)
print(f"Final clean dataset: {df_clean.shape}")

Final clean dataset: (12801, 19)


In [110]:
df_clean

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,source_file,pull_type,brightness,bright_t31,type
0,36.51710,4.27820,NaN,1.60,1.20,2021-08-09,2207,Terra,MODIS,100,6.03,NaN,43.80,N,firms_modis_sp_kabylie_aug_2021.csv,historical,331.40,303.50,0.0
1,36.79965,10.28588,308.70,0.43,0.38,2021-08-09,57,N,VIIRS,n,2,297.97,4.04,N,firms_viirs_snpp_sp_kabylie_aug_2021.csv,historical,NaN,NaN,0.0
2,36.79676,10.28367,313.10,0.43,0.38,2021-08-09,57,N,VIIRS,n,2,297.87,4.31,N,firms_viirs_snpp_sp_kabylie_aug_2021.csv,historical,NaN,NaN,0.0
3,36.74711,6.25637,319.75,0.44,0.46,2021-08-09,57,N,VIIRS,n,2,295.92,1.80,N,firms_viirs_snpp_sp_kabylie_aug_2021.csv,historical,NaN,NaN,0.0
4,36.73005,6.25382,305.52,0.44,0.46,2021-08-09,57,N,VIIRS,n,2,294.85,0.78,N,firms_viirs_snpp_sp_kabylie_aug_2021.csv,historical,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12796,36.17724,5.57149,314.48,0.40,0.37,2026-07-04,145,N,VIIRS,n,2.0NRT,287.43,1.27,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
12797,36.17735,3.74931,303.77,0.38,0.36,2026-07-04,145,N,VIIRS,n,2.0NRT,287.64,1.24,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
12798,36.18064,3.75011,319.70,0.38,0.36,2026-07-04,145,N,VIIRS,n,2.0NRT,290.60,1.24,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
12799,35.90964,0.09219,NaN,1.14,1.06,2026-07-04,1516,Aqua,MODIS,56,6.1NRT,NaN,9.42,D,firms_modis_nrt_nrt_20260704.csv,nrt,326.86,312.90,NaN


# EDA: fire season pattern
####  Algeria fire season peaks in July-September


In [111]:
df_clean["year"]  = df_clean["acq_date"].dt.year
df_clean["month"] = df_clean["acq_date"].dt.month

print("Detections per year:")
print(df_clean.groupby("year").size().to_string())

print("\nDetections per month (across all years):")
print(df_clean.groupby("month").size().to_string())

Detections per year:
year
2021    8143
2022    1400
2023    1626
2026    1632

Detections per month (across all years):
month
6     254
7    3004
8    9543


In [112]:
fire_season = df_clean[df_clean["month"].isin([7, 8, 9])]
pct = 100 * len(fire_season) / len(df_clean)
print(f"\nFire season Jul-Sep: {len(fire_season)} detections ({pct:.1f}% of total)")

if pct >= 60:
    print("Confirms expected Algerian fire seasonality")
else:
    print("Lower than expected — check if date ranges cover full fire seasons")


Fire season Jul-Sep: 12547 detections (98.0% of total)
Confirms expected Algerian fire seasonality


#  EDA: fire intensity (FRP)
#### FRP = Fire Radiative Power in megawatts
#### Higher FRP = more intense fire = more biomass burning
#### The Aug 2021 Kabylie fires should dominate the top FRP values

In [113]:
print("FRP (MW) statistics:")
print(df_clean["frp"].describe().round(2))

print("\nTop 10 most intense detections:")
top10 = df_clean.nlargest(10, "frp")[
    ["acq_date", "latitude", "longitude", "frp", "confidence", "instrument", "source_file"]
]
print(top10.to_string(index=False))

FRP (MW) statistics:
count    12801.00
mean        50.70
std        143.01
min          0.00
25%          3.75
50%         11.97
75%         41.03
max       3072.20
Name: frp, dtype: float64

Top 10 most intense detections:
  acq_date  latitude  longitude    frp confidence instrument                         source_file
2021-08-10   36.5968     4.3032 3072.2         75      MODIS firms_modis_sp_kabylie_aug_2021.csv
2021-08-10   36.5725     4.1610 3006.6        100      MODIS firms_modis_sp_kabylie_aug_2021.csv
2021-08-12   36.7045     8.6872 2607.9        100      MODIS firms_modis_sp_kabylie_aug_2021.csv
2021-08-10   36.5812     4.2781 2558.6        100      MODIS firms_modis_sp_kabylie_aug_2021.csv
2021-08-10   36.6164     4.2672 2368.1         75      MODIS firms_modis_sp_kabylie_aug_2021.csv
2021-08-10   36.5693     4.1672 2339.4        100      MODIS firms_modis_sp_kabylie_aug_2021.csv
2021-08-12   36.3867     1.8672 2241.3        100      MODIS firms_modis_sp_kabylie_aug_2021.csv


# Cross-check: Aug 2021 Tizi Ouzou/Béjaïa fires should show hotspots
#### concentrated in NE Algeria (Kabylie region)
#### Approximate bounding box: lat 36.0-37.0, lon 3.5-5.5

In [114]:
aug2021 = df_clean[
    (df_clean["acq_date"] >= "2021-08-09") &
    (df_clean["acq_date"] <= "2021-08-14")
]

print(f"Aug 2021 detections: {len(aug2021)}")

if not aug2021.empty:
    in_kabylie = aug2021[
        aug2021["latitude"].between(35.5, 37.2) &
        aug2021["longitude"].between(3.0, 6.0)
    ]
    pct = 100 * len(in_kabylie) / len(aug2021)
    print(f"In Kabylie bbox: {len(in_kabylie)} ({pct:.1f}%)")
    print(f"Lat range: {aug2021['latitude'].min():.2f} → {aug2021['latitude'].max():.2f}")
    print(f"Lon range: {aug2021['longitude'].min():.2f} → {aug2021['longitude'].max():.2f}")
    
    if pct >= 50:
        print("Majority of detections in expected Kabylie region")
    else:
        print("Less than 50% in Kabylie — verify event dates/coordinates")
else:
    print("No Aug 2021 detections found — check historical pull")

Aug 2021 detections: 8143
In Kabylie bbox: 5612 (68.9%)
Lat range: 27.17 → 37.05
Lon range: -8.50 → 10.79
Majority of detections in expected Kabylie region


In [115]:
df_clean = df_clean.drop(columns=["month","year"])
df_clean

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,source_file,pull_type,brightness,bright_t31,type
0,36.51710,4.27820,NaN,1.60,1.20,2021-08-09,2207,Terra,MODIS,100,6.03,NaN,43.80,N,firms_modis_sp_kabylie_aug_2021.csv,historical,331.40,303.50,0.0
1,36.79965,10.28588,308.70,0.43,0.38,2021-08-09,57,N,VIIRS,n,2,297.97,4.04,N,firms_viirs_snpp_sp_kabylie_aug_2021.csv,historical,NaN,NaN,0.0
2,36.79676,10.28367,313.10,0.43,0.38,2021-08-09,57,N,VIIRS,n,2,297.87,4.31,N,firms_viirs_snpp_sp_kabylie_aug_2021.csv,historical,NaN,NaN,0.0
3,36.74711,6.25637,319.75,0.44,0.46,2021-08-09,57,N,VIIRS,n,2,295.92,1.80,N,firms_viirs_snpp_sp_kabylie_aug_2021.csv,historical,NaN,NaN,0.0
4,36.73005,6.25382,305.52,0.44,0.46,2021-08-09,57,N,VIIRS,n,2,294.85,0.78,N,firms_viirs_snpp_sp_kabylie_aug_2021.csv,historical,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12796,36.17724,5.57149,314.48,0.40,0.37,2026-07-04,145,N,VIIRS,n,2.0NRT,287.43,1.27,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
12797,36.17735,3.74931,303.77,0.38,0.36,2026-07-04,145,N,VIIRS,n,2.0NRT,287.64,1.24,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
12798,36.18064,3.75011,319.70,0.38,0.36,2026-07-04,145,N,VIIRS,n,2.0NRT,290.60,1.24,N,firms_viirs_snpp_nrt_nrt_20260704.csv,nrt,NaN,NaN,NaN
12799,35.90964,0.09219,NaN,1.14,1.06,2026-07-04,1516,Aqua,MODIS,56,6.1NRT,NaN,9.42,D,firms_modis_nrt_nrt_20260704.csv,nrt,326.86,312.90,NaN


In [116]:
PROC_DIR = Path("../data/Response/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

out_path = PROC_DIR / "firms_algeria_clean.csv"
df_clean.to_csv(out_path, index=False)

print(f"Saved → {out_path}")
print(f"   Shape: {df_clean.shape}")
print(f"   Date range: {df_clean['acq_date'].min()} → {df_clean['acq_date'].max()}")

Saved → ../data/Response/processed/firms_algeria_clean.csv
   Shape: (12801, 19)
   Date range: 2021-08-09 00:00:00 → 2026-07-04 00:00:00
